# IVF Pregnancy Prediction Pipeline

## Overview
- EDA 기반 Feature Engineering
- Data Leakage 방지 전처리
- LightGBM + Optuna 하이퍼파라미터 튜닝
- Submission 파일 생성

---

## Pipeline
1. 데이터 로드
2. 전처리
3. Feature Engineering
4. 모델 학습 (Optuna)
5. 예측 및 제출 파일 생성

In [ ]:
!apt-get update -qq
!apt-get install -y fonts-nanum -qq

In [ ]:
import pandas as pd
import numpy as np

import lightgbm as lgb
import optuna
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# =========================
# 한글 폰트 (캐글 기준)
# =========================
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
font = fm.FontProperties(fname=font_path).get_name()
plt.rc('font', family=font)
plt.rcParams['axes.unicode_minus'] = False

# Optuna 로그 제거
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 데이터 로드

In [ ]:
train = pd.read_csv("/kaggle/input/datasets/mkim98/fertility-dataset/train.csv")
test = pd.read_csv("/kaggle/input/datasets/mkim98/fertility-dataset/test.csv")

TARGET = "임신 성공 여부"
ID_COL = "ID"

print(train.shape, test.shape)
train.head()

## 전처리 함수 정의
- Data Leakage 방지
- Train 기준 encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

def drop_columns(df):
    drop_cols = []

    if ID_COL in df.columns:
        drop_cols.append(ID_COL)

    redundant_cols = ["파트너 정자와 혼합된 난자 수"]

    for col in redundant_cols:
        if col in df.columns:
            drop_cols.append(col)

    return df.drop(columns=drop_cols, errors="ignore")


def convert_str_to_numeric(df):
    age_map = {
        "만18-34세": 26,
        "만35-37세": 36,
        "만38-39세": 38.5,
        "만40-42세": 41,
        "만43-44세": 43.5,
        "만45-50세": 47,
        "알 수 없음": np.nan
    }
    if "시술 당시 나이" in df.columns:
        df["시술 당시 나이"] = df["시술 당시 나이"].map(age_map)
    return df


def handle_missing(df):
    zero_fill_cols = [
        "PGS 시술 여부", "PGD 시술 여부",
        "착상 전 유전 검사 사용 여부",
        "배아 해동 경과일", "난자 해동 경과일"
    ]

    for col in zero_fill_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    num_cols = df.select_dtypes(include=["number"]).columns
    for col in num_cols:
        if col != TARGET:
            df[col] = df[col].fillna(df[col].median())

    cat_cols = df.select_dtypes(include=["object"]).columns
    for col in cat_cols:
        df[col] = df[col].astype(str).replace(['nan','None'], 'Unknown')

    return df


def create_features(df):
    num_fix_cols = [
        "총 생성 배아 수", "혼합된 난자 수",
        "이식된 배아 수", "총 임신 횟수", "총 시술 횟수"
    ]

    for col in num_fix_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df["배아_생성_효율"] = df["총 생성 배아 수"] / (df["혼합된 난자 수"] + 1)
    df["이식_효율"] = df["이식된 배아 수"] / (df["총 생성 배아 수"] + 1)
    df["과거_임신_성공_비율"] = df["총 임신 횟수"] / (df["총 시술 횟수"] + 1)

    df["전체_시술_효율"] = df["배아_생성_효율"] * df["이식_효율"]

    if "시술 당시 나이" in df.columns:
        df["고령_여부"] = (df["시술 당시 나이"] >= 38).astype(int)
        df["나이_이식_상호작용"] = df["시술 당시 나이"] * df["이식된 배아 수"]
        df["난소_반응성_지표"] = df["혼합된 난자 수"] / (df["시술 당시 나이"] + 1)
        df["고령_다배아이식"] = ((df["시술 당시 나이"] >= 38) & (df["이식된 배아 수"] >= 3)).astype(int)

    df["과배란_위험"] = (df["혼합된 난자 수"] > 20).astype(int)
    df["최적_이식_여부"] = df["이식된 배아 수"].between(1, 2).astype(int)

    return df


def encode_categorical(train_df, test_df):
    cat_cols = train_df.select_dtypes(include=["object"]).columns

    for col in cat_cols:
        if col == TARGET:
            continue

        le = LabelEncoder()
        train_df[col] = le.fit_transform(train_df[col].astype(str))

        mapping = {k: v for v, k in enumerate(le.classes_)}
        test_df[col] = test_df[col].astype(str).map(mapping).fillna(-1).astype(int)

    return train_df, test_df


def preprocess(train, test):
    train = drop_columns(train)
    test = drop_columns(test)

    train = convert_str_to_numeric(train)
    test = convert_str_to_numeric(test)

    train = handle_missing(train)
    test = handle_missing(test)

    train = create_features(train)
    test = create_features(test)

    train, test = encode_categorical(train, test)

    return train, test

## 전처리 실행

In [ ]:
train_df, test_df = preprocess(train, test)

X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]

print(X.shape, test_df.shape)

## Optuna + LightGBM 모델 학습

In [ ]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,

        "n_estimators": trial.suggest_int("n_estimators", 300, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.08),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "max_depth": trial.suggest_int("max_depth", 3, 10),

        "min_child_samples": trial.suggest_int("min_child_samples", 20, 80),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    aucs = []

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )

        preds = model.predict_proba(X_val)[:, 1]
        aucs.append(roc_auc_score(y_val, preds))

    return np.mean(aucs)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best Params:", study.best_params)

## CV 점수 (예상)

In [ ]:
cv_scores = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for tr_idx, val_idx in skf.split(X, y):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**study.best_params, verbosity=-1)

    model.fit(X_tr, y_tr)

    preds = model.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y_val, preds)
    cv_scores.append(score)

print("CV Scores:", cv_scores)
print("Mean CV AUC:", np.mean(cv_scores))

## 최종 모델 학습 + Feature Importance

In [ ]:
best_model = lgb.LGBMClassifier(**study.best_params, verbosity=-1)
best_model.fit(X, y)

importance = best_model.feature_importances_

feat_imp = pd.DataFrame({
    "feature": X.columns,
    "importance": importance
}).sort_values(by="importance", ascending=False)

print(feat_imp.head(20))

plt.figure(figsize=(10,6))
plt.barh(feat_imp["feature"].head(20), feat_imp["importance"].head(20))
plt.gca().invert_yaxis()
plt.show()

## Feature Selection + 재학습

In [ ]:
threshold = feat_imp["importance"].quantile(0.2)

low_features = feat_imp[feat_imp["importance"] < threshold]["feature"].tolist()

X_reduced = X.drop(columns=low_features)
test_reduced = test_df.drop(columns=low_features)

best_model = lgb.LGBMClassifier(**study.best_params, verbosity=-1)
best_model.fit(X_reduced, y)

print("✅ Feature Selection 완료")

## 예측 및 제출 파일 생성

In [ ]:
preds = best_model.predict_proba(test_reduced)[:, 1]

submission = pd.DataFrame({
    "ID": test["ID"],
    "probability": preds
})

submission.to_csv("submission.csv", index=False)

print("✅ submission.csv 생성 완료")
submission.head()